# Notebook for NN Library merging (RQ3)

Test notebook for neural network Shapley value benchmarks. Uses the test config
(`config-test-nn-RQ3.yaml`) with minimal settings for fast iteration.

In [1]:
# Auto-reload edited modules (Benchmarking/, Models/, ...) before each cell runs,
# so editing a .py file takes effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config, load_seed, as_list
from Models.dataset_and_models import Dataset, Model
import pandas as pd

In [3]:
# Load configurations — use test config for fast iteration, swap to full config for real runs
CONFIG = "../configs/config-test-nn-RQ3.yaml"
# CONFIG = "../configs/config-neural-networks-RQ3.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}
seed = load_seed(CONFIG)

In [4]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [ ]:
import yaml
from Benchmarking.backends import (
    ShapIQTrueValueBackend,
    CaptumApproxBackend,
    ShapNNApproxBackend,
    LightShapApproxBackend,
    DalexApproxBackend,
)

with open(CONFIG) as f:
    _bench = yaml.safe_load(f)["benchmark"]

# seed may be a scalar or a list; every list value is an extra sweep dimension, so a
# (model, dataset) pair becomes one cell per seed (mirrors slurm/run_benchmark.py).
_n_seeds = len(as_list(_bench["seed"]))
n_cells = len(model_runs) * len(dataset_runs) * _n_seeds

_backend_by_lib = {
    "captum": CaptumApproxBackend,
    "shap_nn": ShapNNApproxBackend,
    "lightshap": LightShapApproxBackend,
    "dalex": DalexApproxBackend,
}

n_specs = len(_bench["budgets"]) * sum(
    sum(1 for appr in _bench["approximators"]
        if appr in getattr(_backend_by_lib[lib], "SUPPORTED_APPROXIMATORS", _bench["approximators"]))
    for lib in _bench["libraries"]
)
n_true_value = len(_bench.get("nn_true_value_libraries", []))
runs_per_cell = n_true_value + n_specs
total_runs = n_cells * runs_per_cell

print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs × {_n_seeds} seed = {n_cells} (model, dataset, seed) cells")
print(
    f"per cell: {n_true_value} true-value (shapiq) + {n_specs} approx specs "
    f"({len(_bench['libraries'])} libs × {len(_bench['approximators'])} approximators "
    f"× {len(_bench['budgets'])} budgets, minus unsupported slots) "
    f"= {runs_per_cell} backend runs"
)
print(f"→ {n_cells} × {runs_per_cell} = {total_runs} total backend runs / results.csv rows")

### Iterate trough every combination and train the model for explanation

In [ ]:
import warnings
import yaml
from tqdm import tqdm
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import (
    ShapIQTrueValueBackend,
    CaptumApproxBackend,
    ShapNNApproxBackend,
    LightShapApproxBackend,
    DalexApproxBackend,
)

warnings.filterwarnings(
    "ignore",
    message="Not all budget.*",
    category=UserWarning,
    module="shapiq",
)
warnings.filterwarnings(
    "ignore",
    message="The sample size is larger.*",
    category=UserWarning,
    module="shapiq",
)

APPROX_BACKEND_BY_LIBRARY = {
    "captum": CaptumApproxBackend,
    "shap_nn": ShapNNApproxBackend,
    "lightshap": LightShapApproxBackend,
    "dalex": DalexApproxBackend,
}

NN_TRUE_VALUE_MAP = {
    "shapiq": ShapIQTrueValueBackend,
}

with open(CONFIG) as f:
    benchmark_config = yaml.safe_load(f)["benchmark"]
n_background = benchmark_config["n_background"]
n_eval = benchmark_config["n_eval"]
# seed may be a scalar or a list; as_list normalizes it so we can sweep it as an
# extra outer loop dimension, exactly like slurm/run_benchmark.py and library_merge.ipynb.
seeds = as_list(benchmark_config["seed"])
imputer = benchmark_config["imputer"]
BUDGETS = benchmark_config["budgets"]
APPROXIMATORS = benchmark_config["approximators"]
device = benchmark_config.get("device", "cpu")

TRUE_VALUE_BACKENDS = [
    NN_TRUE_VALUE_MAP[lib]
    for lib in benchmark_config.get("nn_true_value_libraries", [])
    if lib in NN_TRUE_VALUE_MAP
]

approximation_specs = [
    (APPROX_BACKEND_BY_LIBRARY[lib], {"approximator": appr, "budget": bgt})
    for lib in benchmark_config["libraries"]
    for appr in APPROXIMATORS
    for bgt in BUDGETS
    if appr in getattr(APPROX_BACKEND_BY_LIBRARY[lib], "SUPPORTED_APPROXIMATORS", APPROXIMATORS)
]

RESULTS_CSV = "../Benchmarking/results_nn.csv"

total_weight = len(seeds) * sum(
    dataset_params.get("n_features", 1)
    for _, dataset_params in dataset_runs
    for _ in model_runs
)

with tqdm(total=total_weight, desc="benchmark", unit="feat") as bar:
    for seed in seeds:
        for dataset_key, dataset_params in dataset_runs:
            dataset_enum = Dataset[dataset_key.upper()]
            ds = dataset_enum.load_dataset(**dataset_params, seed=seed)
            X, y = ds["X"], ds["y"]
            weight = dataset_params.get("n_features", 1)

            for model_key, model_params in model_runs:
                bar.set_postfix_str(
                    f"{dataset_key} nf={dataset_params.get('n_features')} | {model_key} | seed={seed}"
                )
                trainer = Model[model_key.upper()].get_model_with_params(
                    dataset_enum, {**model_params, "device": device}, seed=seed
                )
                trainer.fit(X, y, task=ds["task"])

                # Rebuilt per cell so the runner's RNG matches the current seed.
                runner = BenchmarkRunner(
                    true_value_backends=TRUE_VALUE_BACKENDS,
                    approximation_specs=approximation_specs,
                    output_csv=RESULTS_CSV,
                    n_background=n_background,
                    n_eval=n_eval,
                    seed=seed,
                    imputer=imputer,
                )
                runner.run(
                    model=trainer.get_model(),
                    X=X,
                    run_meta={
                        "dataset": dataset_key,
                        "model": model_key,
                        "order": 1,
                        "n_features": dataset_params.get("n_features"),
                        "n_samples": dataset_params.get("n_samples"),
                        "seed": seed,
                    },
                )
                bar.update(weight)

print(f"Done. Results written to {RESULTS_CSV}")

In [ ]:
results = pd.read_csv(RESULTS_CSV)
n_before = len(results)

# seed is part of the cell key so swept seeds stay distinct (mirrors library_merge.ipynb).
results = results.drop_duplicates(
    subset=["dataset", "model", "n_features", "n_samples", "seed",
            "backend", "approximator", "budget"],
    keep="last",
)

results.to_csv(RESULTS_CSV, index=False)
print(f"de-duplicated {n_before} -> {len(results)} rows; overwrote {RESULTS_CSV}")

In [ ]:
print(f"{len(results)} rows written")

cols = [
    "dataset", "model", "n_features", "backend", "approximator", "budget",
    "n_model_evals", "runtime_s", "mean_abs_diff", "relative_mae",
    "sign_agreement", "mean_sample_rho",
]
results[cols].head(20)

results